# 03 — Preprocesamiento (con control de fuga)

## Introducción

**Objetivo.** Definir features, codificación, escalado y splits evitando fuga de información.

**Fundamento teórico.** Estandarización (z-score), one-hot para nominales y separación train/test estratificada son prácticas estándar; el control de fuga es lo distintivo aquí.

**Ventajas.** Pipeline reproducible; impide que información del test contamine el entrenamiento.

**Limitaciones.** El one-hot aumenta dimensionalidad; el escalado asume relaciones aproximadamente lineales para algunos modelos.

**Casos de uso.** Base común para entrenar y comparar todos los modelos.


In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath('../src'))
import warnings; warnings.simplefilter('ignore')
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
from predia_ml import config, data, evaluate, plots
pd.set_option('display.max_columns', 60)


### Taxonomía de variables (decisión central del proyecto)
- **Fuga directa (se eliminan siempre):** `diabetes_stage`, `diabetes_risk_score`.
- **Laboratorios diagnósticos (excluidos en *screening*):** `hba1c`, `glucose_fasting`, `glucose_postprandial`, `insulin_level`.
- **Marco *screening* (honesto, producción):** demografía, estilo de vida, antropometría, presión, lípidos, antecedentes.
- **Marco *clinical*:** *screening* + laboratorios diagnósticos (más preciso, fuga parcial).

In [2]:
from predia_ml import preprocess
print('Screening features:', config.feature_columns('screening'))
print('\nClinical features:', config.feature_columns('clinical'))
print('\nFUGA (drop):', config.LEAKY_COLS)

Screening features: ['gender', 'ethnicity', 'education_level', 'income_level', 'employment_status', 'smoking_status', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history', 'age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides']

Clinical features: ['gender', 'ethnicity', 'education_level', 'income_level', 'employment_status', 'smoking_status', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history', 'age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'hba1c', 'glucose_fastin

In [3]:
df = data.load_raw()
X, y = preprocess.make_xy(df, 'screening')
prep = preprocess.build_preprocessor('screening')
Xt = prep.fit_transform(X)
print('Matriz transformada:', Xt.shape)
print('Ejemplo de features de salida:', list(prep.get_feature_names_out())[:12])

Matriz transformada: (100000, 42)
Ejemplo de features de salida: ['gender_Female', 'gender_Male', 'gender_Other', 'ethnicity_Asian', 'ethnicity_Black', 'ethnicity_Hispanic', 'ethnicity_Other', 'ethnicity_White', 'education_level_Graduate', 'education_level_Highschool', 'education_level_No formal', 'education_level_Postgraduate']


Los splits estratificados y reproducibles (seed=42) se guardan en `datasets/split_*.joblib` y se reutilizan en todos los notebooks de modelo.